### LIMPIEZA DE ETIQUETAS THINK

In [ ]:
import pandas as pd
import json
import os

def limpiar_y_filtrar():
    archivo_entrada = 'dataset/dataset_final_records.json'
    archivo_salida_limpio = 'dataset/dataset_final_limpio.json'
    archivo_salida_think = 'dataset/registros_think.json'
    
    print("Iniciando procesamiento...")

    if not os.path.exists(archivo_entrada):
        print(f"Error: No se encontró '{archivo_entrada}'.")
        return

    # 1. Cargar datos
    df = pd.read_json(archivo_entrada, orient='records')
    total_registros = len(df)
    print(f"Dataset cargado: {total_registros} registros listos para analizar.\n")

    # Contadores
    registros_limpiados = 0
    registros_con_think = 0
    lista_ids_think = []

    print("Iniciando limpieza y búsqueda (mostrando progreso)...")
    print("-" * 50)

    # 2. Función iterativa para limpiar y buscar
    for index, fila in df.iterrows():
        texto_original = str(fila['comentario'])
        
        # --- FASE DE LIMPIEZA ---
        # 1. Reemplazar saltos de línea reales y literales
        texto_limpio = texto_original.replace('\n', ' ').replace('\\n', ' ')
        # 2. Eliminar combinaciones de comillas escapadas (el caso de \"pensaste\")
        texto_limpio = texto_limpio.replace('\\"', '"')
        # 3. Eliminar diagonales y diagonales invertidas
        texto_limpio = texto_limpio.replace('\\/', '').replace('/', '').replace('\\', '')
        # 4. Quitar espacios múltiples
        texto_limpio = ' '.join(texto_limpio.split())

        # Si el texto cambió, sumamos al contador general de limpieza
        if texto_original != texto_limpio:
            df.at[index, 'comentario'] = texto_limpio
            registros_limpiados += 1

        # --- FASE DE BÚSQUEDA <think> ---
        # Buscamos la etiqueta en el texto ya limpio
        if '<think>' in texto_limpio.lower():
            registros_con_think += 1
            lista_ids_think.append(fila['id'])
            # Mostramos en tiempo real cuando encontramos uno
            print(f" ¡Etiqueta <think> encontrada! -> ID: {fila['id']} | Acumulados: {registros_con_think}")

        # Mostrar progreso cada 500 registros para saber que el script sigue corriendo
        if (index + 1) % 500 == 0:
            print(f"Progreso: {index + 1} / {total_registros} analizados...")

    print("-" * 50)

    # 3. Separar los registros que tienen <think>
    # Filtramos el DataFrame usando la lista de IDs que encontramos
    df_think = df[df['id'].isin(lista_ids_think)]

    # 4. Guardar archivos
    # Guardamos el dataset completo ya limpio
    df.to_json(archivo_salida_limpio, orient='records', force_ascii=False, indent=4)
    
    # Guardamos un archivo EXCLUSIVO con los que tienen <think> (si es que hay)
    if not df_think.empty:
        df_think.to_json(archivo_salida_think, orient='records', force_ascii=False, indent=4)

    # 5. Resumen final en consola
    print("\n--- RESUMEN FINAL ---")
    print(f"Total analizado: {total_registros} registros.")
    print(f"Registros sucios corregidos: {registros_limpiados}")
    print(f"Total de comentarios con <think>: {registros_con_think}")
    if registros_con_think > 0:
        print(f"IDs con <think>: {lista_ids_think}")
        print(f"Se guardó un archivo aislado con estos registros en: {archivo_salida_think}")
    print(f"Dataset principal limpio guardado en: {archivo_salida_limpio}")
    print("-------------------------\n")

if __name__ == "__main__":
    limpiar_y_filtrar()

🚀 Iniciando procesamiento...
✅ Dataset cargado: 2500 registros listos para analizar.

🧹 Iniciando limpieza y búsqueda (mostrando progreso)...
--------------------------------------------------
⏳ Progreso: 500 / 2500 analizados...
⏳ Progreso: 1000 / 2500 analizados...
  🚨 ¡Etiqueta <think> encontrada! -> ID: 1170 | Acumulados: 1
  🚨 ¡Etiqueta <think> encontrada! -> ID: 1480 | Acumulados: 2
⏳ Progreso: 1500 / 2500 analizados...
  🚨 ¡Etiqueta <think> encontrada! -> ID: 1749 | Acumulados: 3
  🚨 ¡Etiqueta <think> encontrada! -> ID: 1821 | Acumulados: 4
⏳ Progreso: 2000 / 2500 analizados...
⏳ Progreso: 2500 / 2500 analizados...
--------------------------------------------------

📊 --- RESUMEN FINAL ---
Total analizado: 2500 registros.
Registros sucios corregidos: 445
Total de comentarios con <think>: 4
IDs con <think>: [1170, 1480, 1749, 1821]
💾 Se guardó un archivo aislado con estos registros en: dataset/registros_think.json
💾 Dataset principal limpio guardado en: dataset/dataset_final_limp

### *Revisión final de etiqueta think*

In [ ]:
import pandas as pd
import json
import os

def limpiar_y_filtrar():
    archivo_entrada = 'dataset/dataset_final_records.json'
    archivo_salida_limpio = 'dataset/dataset_final_limpio.json'
    archivo_salida_think = 'dataset/registros_think.json'
    
    print("Iniciando procesamiento...")

    if not os.path.exists(archivo_entrada):
        print(f"Error: No se encontró '{archivo_entrada}'.")
        return

    # 1. Cargar datos
    df = pd.read_json(archivo_entrada, orient='records')
    total_registros = len(df)
    print(f"Dataset cargado: {total_registros} registros listos para analizar.\n")

    registros_limpiados = 0
    registros_con_think = 0
    lista_ids_think = []

    print("Iniciando limpieza extrema (mostrando progreso)...")
    print("-" * 50)

    for index, fila in df.iterrows():
        texto_original = str(fila['comentario'])
        
        # --- FASE DE LIMPIEZA EXTREMA ---
        # 1. Saltos de línea
        texto_limpio = texto_original.replace('\n', ' ').replace('\\n', ' ')
        
        # 2. SOLUCIÓN AL PROBLEMA DE ESCAPE JSON:
        # Cambiamos comillas dobles (") por simples (') para que JSON no genere los \"
        texto_limpio = texto_limpio.replace('"', "'")
        
        # 3. Eliminar cualquier diagonal o barra invertida residual
        texto_limpio = texto_limpio.replace('\\/', '').replace('/', '').replace('\\', '')
        
        # 4. Quitar espacios múltiples
        texto_limpio = ' '.join(texto_limpio.split())

        # Actualizar si hubo cambios
        if texto_original != texto_limpio:
            df.at[index, 'comentario'] = texto_limpio
            registros_limpiados += 1

        # --- FASE DE BÚSQUEDA <think> ---
        if '<think>' in texto_limpio.lower():
            registros_con_think += 1
            lista_ids_think.append(fila['id'])
            print(f"¡Etiqueta <think> encontrada! -> ID: {fila['id']} | Acumulados: {registros_con_think}")

        if (index + 1) % 500 == 0:
            print(f"Progreso: {index + 1} / {total_registros} analizados...")

    print("-" * 50)

    # 3. Separar los registros que tienen <think>
    df_think = df[df['id'].isin(lista_ids_think)]

    # 4. Guardar archivos
    df.to_json(archivo_salida_limpio, orient='records', force_ascii=False, indent=4)
    
    if not df_think.empty:
        df_think.to_json(archivo_salida_think, orient='records', force_ascii=False, indent=4)

    # 5. Resumen final en consola
    print("\n--- RESUMEN FINAL ---")
    print(f"Total analizado: {total_registros} registros.")
    print(f"Registros sucios corregidos: {registros_limpiados}")
    print(f"Total de comentarios con <think>: {registros_con_think}")
    if registros_con_think > 0:
        print(f"IDs con <think>: {lista_ids_think}")
        print(f"Archivo aislado con estas etiquetas: {archivo_salida_think}")
    print(f"Dataset principal súper limpio en: {archivo_salida_limpio}")
    print("-------------------------\n")

limpiar_y_filtrar()

🚀 Iniciando procesamiento...
✅ Dataset cargado: 2500 registros listos para analizar.

🧹 Iniciando limpieza extrema (mostrando progreso)...
--------------------------------------------------
⏳ Progreso: 500 / 2500 analizados...
⏳ Progreso: 1000 / 2500 analizados...
⏳ Progreso: 1500 / 2500 analizados...
⏳ Progreso: 2000 / 2500 analizados...
⏳ Progreso: 2500 / 2500 analizados...
--------------------------------------------------

📊 --- RESUMEN FINAL ---
Total analizado: 2500 registros.
Registros sucios corregidos: 674
Total de comentarios con <think>: 0
💾 Dataset principal súper limpio en: dataset/dataset_final_limpio.json
-------------------------



In [17]:
pip install pandas openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Dataset limpio para fase 2 en excell

In [ ]:
import pandas as pd
import json

# Ruta del archivo JSON
ruta = 'dataset/dataset_final_limpio.json'

# Leer el archivo JSON
with open(ruta, 'r', encoding='utf-8') as f:
    data = json.load(f)   # Esto convierte el JSON en una lista de diccionarios

# Convertir JSON a DataFrame
df = pd.DataFrame(data)

# Exportar a Excel
df.to_excel("dataset/dataset_final_limpio.xlsx", index=False)